In [0]:
%pip install tqdm pyarrow transformers torch wandb scikit-learn adjustText --quiet

In [0]:
import sys
import os
import runpy

# Add data_preprocess to path so modules can import each other
sys.path.insert(0, os.path.abspath('data_preprocess'))

# Run each preprocessing step in-process (keeps access to SparkSession)
# Step 1/6: Irregular time series
print("=== Step 1/6: Irregular time series (labs + vitals) ===")
sys.argv = ['preprocess_irg_time_series.py', '--output_dir', './data', '--batch_size', '40000']
runpy.run_path('data_preprocess/preprocess_irg_time_series.py', run_name='__main__')

# Step 2/6: Imputed regular time series
print("=== Step 2/6: Imputed regular time series ===")
sys.argv = ['preprocess_imputed_time_series.py', '--output_dir', './data']
runpy.run_path('data_preprocess/preprocess_imputed_time_series.py', run_name='__main__')

# Step 3/6: Radiology notes text
print("=== Step 3/6: Radiology notes text ===")
sys.argv = ['preprocess_notes.py', '--notes_file_path', '/Volumes/mimiciv/note/data/radiology.csv.gz', '--output_dir', './data']
runpy.run_path('data_preprocess/preprocess_notes.py', run_name='__main__')

# Step 4/6: BioBERT note embeddings (GPU)
print("=== Step 4/6: BioBERT note embeddings ===")
sys.argv = ['preprocess_notes_embeddings.py', '--output_dir', './data', '--device_number', '0']
runpy.run_path('data_preprocess/preprocess_notes_embeddings.py', run_name='__main__')

# Step 5/6: Create IHM task
print("=== Step 5/6: Create IHM task ===")
sys.argv = ['create_ihm_task.py', '--output_dir', './data', '--restrict_hours', '48', '--include_notes', '--include_missing', '--standardize_features', '--seed', '42']
runpy.run_path('data_preprocess/create_ihm_task.py', run_name='__main__')

# Step 6/6: Create LOS task
print("=== Step 6/6: Create LOS task ===")
sys.argv = ['create_los_task.py', '--output_dir', './data', '--include_notes', '--include_missing', '--standardize_features', '--seed', '42']
runpy.run_path('data_preprocess/create_los_task.py', run_name='__main__')

In [0]:
import sys, os, runpy

# Set working directory to mimiciv/
os.chdir('/Workspace/Users/patrick.kasl@bayesianhealth.com/merge/mimiciv')

# Add parent (merge/) to path for model imports, and pid/ for RUS estimator
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

# Run RUS computation for IHM task
print("=== Computing RUS for IHM task ===")
sys.argv = [
    'mimiciv_rus_multimodal.py',
    '--train_dataset_path', './data/ihm/train_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--task', 'ihm',
    '--seq_len', '48',
    '--num_lags', '8',
    '--sequence_pooling', 'mean',
    '--gpu', '0'
]
runpy.run_path('mimiciv_rus_multimodal.py', run_name='__main__')

In [0]:
import sys, os, runpy

os.chdir('/Workspace/Users/patrick.kasl@bayesianhealth.com/merge/mimiciv')
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

print("=== Training IHM multimodal TRUS-MoE ===")
sys.argv = [
    'train_mimiciv_multimodal.py',
    '--train_data_path', './data/ihm/train_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--val_data_path', './data/ihm/val_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--rus_data_path', './results/ihm/rus_multimodal_all_seq48_lags8_meanpool.npy',
    '--task', 'ihm',
    '--truncate_from_end',
    '--seq_len', '48',
    '--gpu', '0',
    '--lambda_u', '1.0',
    '--lambda_r', '1.0',
    '--lambda_s', '1.0',
    '--lambda_load', '0.02',
    '--run_name', 'mimiciv_ihm_lambdarus1.0_lambdaload0.02_seed42',
    '--seed', '42',
    '--lr', '1e-3',
    '--epochs', '20',
    '--output_dir', './results'
]
runpy.run_path('train_mimiciv_multimodal.py', run_name='__main__')

In [0]:
import sys, os, runpy

os.chdir('/Workspace/Users/patrick.kasl@bayesianhealth.com/merge/mimiciv')
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

CHECKPOINT = './results/ihm/checkpoints/mimiciv_ihm_lambdarus1.0_lambdaload0.02_seed42/best_multimodal_model_mimiciv.pth'

print("=== Testing IHM model ===")
sys.argv = [
    'test_mimiciv_multimodal.py',
    '--checkpoint_path', CHECKPOINT,
    '--test_data_path', './data/ihm/test_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--rus_data_path', './results/ihm/rus_multimodal_all_seq48_lags8_meanpool.npy',
    '--gpu', '0',
    '--eval_train',
    '--train_data_path', './data/ihm/train_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--eval_val',
    '--val_data_path', './data/ihm/val_ihm-48-notes-missingInd-standardized_stays.pkl',
    '--plot_expert_activations',
    '--plot_num_samples', '1024',
    '--save_metrics'
]
runpy.run_path('test_mimiciv_multimodal.py', run_name='__main__')